## Étape 1 : Importation des bibliothèques et chargement de l'image
On commence par importer les bibliothèques nécessaires :
- **cv2** (OpenCV) : pour le traitement d'images
- **numpy** : pour la manipulation de tableaux numériques
- **scipy.datasets** : pour charger l'image de test « ascent »


In [ ]:
import cv2                      # OpenCV – traitement d'images
import numpy as np              # NumPy – calcul matriciel
from scipy import datasets      # SciPy – jeux de données intégrés
import matplotlib.pyplot as plt # Matplotlib – affichage graphique

# Chargement de l'image « ascent » fournie par SciPy
# C'est une image en niveaux de gris de 512×512 pixels
i = datasets.ascent()

# Vérification des dimensions de l'image chargée
print(f"Dimensions de l'image : {i.shape}")   # (512, 512)
print(f"Type des pixels       : {i.dtype}")    # uint8 (0-255)


## Étape 2 : Affichage de l'image originale
On utilise **matplotlib** pour visualiser l'image originale.
- `plt.gray()` : palette de gris
- `plt.axis('off')` : masquer les axes


In [ ]:
plt.figure(figsize=(6, 6))   
plt.gray()
plt.grid(False)
plt.axis('off')
plt.title('Image originale (512×512)')
plt.imshow(i)
plt.show()


## Étape 3 : Préparation de l'image pour la convolution
On crée une **copie** de l'image pour y stocker le résultat
de la convolution, puis on récupère ses dimensions.


In [ ]:
i_transformed = np.copy(i)
size_x = i_transformed.shape[0] 
size_y = i_transformed.shape[1] 
print(f"Taille de l'image : {size_x} × {size_y}")


## Étape 4 : Définition des filtres de convolution (noyaux 3×3)

Chaque filtre est une matrice **3×3** qui détecte un type de motif :

| Filtre | Nom | Rôle |
|--------|-----|------|
| `filter1` | **Laplacien** | Détection de contours dans toutes les directions |
| `filter2` | **Sobel horizontal** | Détection des bords **horizontaux** |
| `filter3` | **Sobel vertical** | Détection des bords **verticaux** |


In [ ]:
# Filtre 1 – Laplacien : met en évidence les contours
# dans toutes les directions (haut, bas, gauche, droite).
filter1 = [
    [ 0,  1,  0],
    [ 1, -4,  1],
    [ 0,  1,  0]
]

# Filtre 2 – Sobel horizontal : détecte les contours
# horizontaux (gradient vertical).
filter2 = [
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
]

# Filtre 3 – Sobel vertical : détecte les contours
# verticaux (gradient horizontal).
filter3 = [
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
]

# Liste des filtres avec leurs noms pour l'affichage
filters_with_names = [
    (filter1, 'Laplacien'),
    (filter2, 'Sobel horizontal'),
    (filter3, 'Sobel vertical'),
]

print('3 filtres définis avec succès.')


## Étape 5 : Application des 3 filtres – Convolution + Max Pooling

Pour **chaque** filtre on effectue deux opérations :

1. **Convolution** : on parcourt chaque pixel (en excluant les bords)
   et on calcule la somme pondérée du voisinage 3×3 avec le noyau.
   La valeur résultante est bornée entre 0 et 255 (clipping).

2. **Max Pooling 2×2** : on réduit la résolution de moitié en ne gardant
   que la **valeur maximale** de chaque bloc 2×2.  
   L'image passe ainsi de **512×512** à **256×256**.

Les deux résultats (avant et après pooling) sont affichés
côte à côte pour chaque filtre.


In [ ]:
# ============================================================
# Boucle sur les 3 filtres : Convolution + Affichage + Pooling
# ============================================================

poids = 1.0  # Poids multiplicatif appliqué après la convolution

for flr, nom in filters_with_names:

    # ── 1) Réinitialiser la copie de l'image pour chaque filtre ──
    i_transformed = np.copy(i)

    # ── 2) CONVOLUTION ──────────────────────────────────────────
    # On parcourt tous les pixels sauf la bordure de 1 pixel
    # (car le noyau 3×3 déborderait de l'image).
    for x in range(1, size_x - 1):
        for y in range(1, size_y - 1):
            out = 0.0  # Accumulateur (float pour gérer les valeurs négatives)

            # Produit élément par élément entre le voisinage 3×3
            # du pixel (x,y) et le filtre, puis somme.
            # Ligne du haut du noyau (flr[0])
            out += float(i[x - 1, y - 1]) * flr[0][0]   # haut-gauche
            out += float(i[x    , y - 1]) * flr[0][1]   # haut-centre
            out += float(i[x + 1, y - 1]) * flr[0][2]   # haut-droite

            # Ligne du milieu du noyau (flr[1])
            out += float(i[x - 1, y    ]) * flr[1][0]   # milieu-gauche
            out += float(i[x    , y    ]) * flr[1][1]   # centre (pixel courant)
            out += float(i[x + 1, y    ]) * flr[1][2]   # milieu-droite

            # Ligne du bas du noyau (flr[2])
            out += float(i[x - 1, y + 1]) * flr[2][0]   # bas-gauche
            out += float(i[x    , y + 1]) * flr[2][1]   # bas-centre
            out += float(i[x + 1, y + 1]) * flr[2][2]   # bas-droite

            # Application du poids
            out = out * poids

            # Clipping : borner la valeur entre 0 et 255
            if out < 0:
                out = 0
            if out > 255:
                out = 255

            # Stocker le résultat dans l'image transformée
            i_transformed[x, y] = int(out)

    # ── 3) AFFICHAGE après convolution (512×512) ────────────────
    # ── 4) MAX POOLING 2×2 ──────────────────────────────────────
    # On réduit l'image de moitié en gardant le max de chaque bloc 2×2.
    new_x = int(size_x / 2)   # 256
    new_y = int(size_y / 2)   # 256
    newImage = np.zeros((new_x, new_y))  # Image réduite

    for x in range(0, size_x, 2):        # Pas de 2 en x
        for y in range(0, size_y, 2):    # Pas de 2 en y
            # Récupérer les 4 pixels du bloc 2×2
            pixels = [
                i_transformed[x    , y    ],
                i_transformed[x + 1, y    ],
                i_transformed[x    , y + 1],
                i_transformed[x + 1, y + 1],
            ]
            # Garder la valeur maximale (Max Pooling)
            newImage[int(x / 2), int(y / 2)] = max(pixels)

    # ── 5) AFFICHAGE côte à côte ────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Image après convolution (512×512)
    axes[0].imshow(i_transformed, cmap='gray')
    axes[0].set_title(f'Convolution – {nom} (512×512)')
    axes[0].axis('off')

    # Image après max pooling (256×256)
    axes[1].imshow(newImage, cmap='gray')
    axes[1].set_title(f'Max Pooling 2×2 – {nom} (256×256)')
    axes[1].axis('off')

    plt.suptitle(f'Filtre : {nom}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'Noyau utilisé : {flr}')
    print('─' * 60)


## Conclusion

Ce TP montre les deux opérations fondamentales d'un **CNN** :

1. **Convolution** : extraction de caractéristiques (contours, textures…)
   grâce à des noyaux 3×3.
2. **Max Pooling** : réduction de la résolution tout en conservant
   les informations les plus saillantes.

En variant les filtres on observe différents types de contours détectés :
- Le **Laplacien** détecte les contours dans toutes les directions.
- Le **Sobel horizontal** met en avant les bords horizontaux.
- Le **Sobel vertical** met en avant les bords verticaux.
